# VRA/ANAC — ELT de 2021
**Projeto de Integração — Banco de Dados · CIn/UFPE**

**Responsável pelo ELT 2021:** Murilo Acioli.
**Base:** notebook `notebooks/elt/elt_2020.ipynb` (Gustavo José Arraes de Queiroz), derivado do ELT 2019 do grupo, no [repositório do projeto](https://github.com/LeonardoBarrosCavalcantiNeto/Projeto-de-integracao-BD).

O ELT primeiro carrega os valores originais no PostgreSQL; só depois o SQL trata duplicatas, campos vazios, datas e métricas.

**Fonte dos dados:** arquivos mensais do VRA publicados pela ANAC em `https://siros.anac.gov.br/siros/registros/diversos/vra/2021/` (`VRA_2021_01.csv` a `VRA_2021_12.csv`), a mesma origem usada para 2020.

**Como usar:** no Google Colab, execute as células na ordem (Ambiente de execução → Executar tudo). A etapa 3 baixa os 12 CSVs direto da ANAC; se preferir, ela também aceita o envio manual de `data_2021_anac.zip`. Não é necessário abrir ou salvar os CSVs no Excel.

**Particularidades de 2021 (diferenças em relação a 2019/2020):**
- **Dois layouts no mesmo ano.** Sete meses (01, 02, 03, 07, 09, 11 e 12) foram republicados pela ANAC com uma 21ª coluna, `Codeshare`, e quebra de linha CRLF. Os outros cinco mantêm as 20 colunas. As 20 primeiras colunas são idênticas ao contrato. `Codeshare` é preservada na camada bruta (`NULL` quando o arquivo não tem a coluna) e **não** entra na comparação de duplicatas, que continua usando as 20 colunas originais do contrato.
- **Dois formatos de `Referência`.** Nos meses com 21 colunas, o valor vem como `DD/MM/AAAA 00:00:00`; nos demais, como `AAAA-MM-DD`. Os dois são convertidos para `DATE`.

**Escopo:** carga bruta, transformação, auditoria e exportação de evidências de 2021. As dimensões e a `fato_voo` compartilhadas entre 2019–2021 ficam para a integração do grupo. Não unir resultados ETL e ELT como se fossem voos distintos.

> O PostgreSQL será temporário, dentro do Colab. Ao encerrar a sessão, o banco pode desaparecer. Baixe as evidências e salve o notebook executado no GitHub.


## 1. Instalar o PostgreSQL
Execute e espere terminar. Isso prepara o servidor e a biblioteca de conexão no Colab.

In [1]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-contrib", "-qq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"], check=True)
subprocess.run(["service", "postgresql", "start"], check=True)
print("PostgreSQL instalado e iniciado no Colab.")

PostgreSQL instalado e iniciado no Colab.


## 2. Conectar ao banco
A senha é temporária e não é exibida. Usaremos apenas o schema `elt_2021`.

In [2]:
import secrets
import psycopg2
import subprocess

senha_pg = secrets.token_urlsafe(24)
# Envía a senha ao psql por stdin (evita registrá-la na linha de comando)
subprocess.run(
    ["runuser", "-u", "postgres", "--", "psql", "-v", "ON_ERROR_STOP=1"],
    input="ALTER USER postgres PASSWORD '" + senha_pg.replace("'", "''") + "';\n",
    text=True, check=True, capture_output=True
)
conn = psycopg2.connect(host="localhost", port=5432, dbname="postgres", user="postgres", password=senha_pg)
conn.autocommit = False
print("Conexão PostgreSQL funcionando!")

Conexão PostgreSQL funcionando!


## 3. Obter os 12 CSVs de 2021
Por padrão, a célula baixa `VRA_2021_01.csv` a `VRA_2021_12.csv` direto do servidor da ANAC (`siros.anac.gov.br`), sem alterar nenhum byte, e monta o ZIP na memória. O SHA-256 de cada arquivo fica no inventário, para comparar com o ETL.

Se o download falhar, mude `BAIXAR_DA_ANAC` para `False`, execute de novo e envie `data_2021_anac.zip` em **Escolher arquivos/Procurar**.


In [3]:
validacao_concluida = False
from google.colab import files
import io
import csv
import zipfile
import hashlib
import re
from pathlib import PurePosixPath
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from psycopg2 import sql

ANO = 2021
SCHEMA = "elt_2021"
VERSAO_PIPELINE = "1.0"
inicio_execucao = datetime.now(timezone.utc).isoformat()

# True: baixa os 12 CSVs direto da ANAC (mesma fonte usada em 2020).
# False: pede o envio manual de data_2021_anac.zip.
BAIXAR_DA_ANAC = True
FONTE_ANAC = "https://siros.anac.gov.br/siros/registros/diversos/vra/2021/"

if BAIXAR_DA_ANAC:
    import urllib.request
    memoria = io.BytesIO()
    with zipfile.ZipFile(memoria, "w", compression=zipfile.ZIP_DEFLATED) as zip_saida:
        for mes in range(1, 13):
            nome_csv = f"VRA_2021_{mes:02d}.csv"
            with urllib.request.urlopen(FONTE_ANAC + nome_csv, timeout=600) as resposta:
                conteudo = resposta.read()
            zip_saida.writestr(nome_csv, conteudo)  # bytes intactos, sem conversão
            print(f"Baixado: {nome_csv} ({len(conteudo):,} bytes)")
    nome_zip, bytes_zip = "data_2021_anac.zip", memoria.getvalue()
else:
    arquivos_enviados = files.upload()
    assert len(arquivos_enviados) == 1, "Envie somente um ZIP com os 12 CSVs de 2021."
    nome_zip, bytes_zip = next(iter(arquivos_enviados.items()))
    assert nome_zip.lower().endswith(".zip"), "O arquivo enviado precisa ser ZIP."
assert zipfile.is_zipfile(io.BytesIO(bytes_zip)), "O conteúdo não é um ZIP válido."
sha256_zip = hashlib.sha256(bytes_zip).hexdigest()
print("ZIP recebido:", nome_zip)
print("SHA-256:", sha256_zip)

Baixado: VRA_2021_01.csv (18,281,868 bytes)
Baixado: VRA_2021_02.csv (13,667,479 bytes)
Baixado: VRA_2021_03.csv (12,444,242 bytes)
Baixado: VRA_2021_04.csv (8,455,519 bytes)
Baixado: VRA_2021_05.csv (10,464,126 bytes)
Baixado: VRA_2021_06.csv (11,928,409 bytes)
Baixado: VRA_2021_07.csv (16,716,773 bytes)
Baixado: VRA_2021_08.csv (15,837,098 bytes)
Baixado: VRA_2021_09.csv (17,466,130 bytes)
Baixado: VRA_2021_10.csv (17,231,843 bytes)
Baixado: VRA_2021_11.csv (19,513,413 bytes)
Baixado: VRA_2021_12.csv (22,010,312 bytes)
ZIP recebido: data_2021_anac.zip
SHA-256: d664772dfc3113a45a60aa67bd4daffb10b00088f9ff85bb6a22a05439020399


## 4. Conferir os 12 arquivos sem modificar os dados
Conferimos nomes, cabeçalhos e número de campos. Cada mês precisa aparecer uma vez. Todos os valores continuam como texto.

`arquivo_origem` identifica o CSV e `linha_origem` indica a linha física onde o registro começa, contando o cabeçalho como linha 1. A leitura respeita campos entre aspas, inclusive quebras de linha dentro deles.

**2021:** são aceitos dois cabeçalhos — as 20 colunas do contrato, ou essas mesmas 20 seguidas de `Codeshare`. Qualquer outra diferença interrompe a execução. O inventário registra quantas colunas cada arquivo tem.


In [4]:
colunas_esperadas = [
    "Sigla ICAO Empresa Aérea", "Empresa Aérea", "Número Voo", "Código DI",
    "Código Tipo Linha", "Modelo Equipamento", "Número de Assentos",
    "Sigla ICAO Aeroporto Origem", "Descrição Aeroporto Origem", "Partida Prevista",
    "Partida Real", "Sigla ICAO Aeroporto Destino", "Descrição Aeroporto Destino",
    "Chegada Prevista", "Chegada Real", "Situação Voo", "Justificativa",
    "Referência", "Situação Partida", "Situação Chegada"
]
# Em 2021, parte dos meses foi republicada pela ANAC com uma 21ª coluna.
COLUNA_EXTRA_2021 = "Codeshare"
layouts_aceitos = {
    20: colunas_esperadas,
    21: colunas_esperadas + [COLUNA_EXTRA_2021],
}

def registros_csv(arquivo_binario, nome):
    """Devolve (linha_origem, 20 campos do contrato, Codeshare ou None)."""
    leitor = csv.reader(io.TextIOWrapper(arquivo_binario, encoding="utf-8-sig", newline=""),
                        delimiter=";", strict=True)
    cabecalho = next(leitor, None)
    if cabecalho is None or layouts_aceitos.get(len(cabecalho)) != cabecalho:
        raise ValueError(f"Cabeçalho diferente do contrato em {nome}: {cabecalho!r}")
    n_campos = len(cabecalho)
    while True:
        inicio_linha = leitor.line_num + 1
        try:
            campos = next(leitor)
        except StopIteration:
            break
        if len(campos) != n_campos:
            raise ValueError(f"{nome}, linha {inicio_linha}: esperados {n_campos} campos, encontrados {len(campos)}.")
        codeshare = campos[20] if n_campos == 21 else None
        yield inicio_linha, campos[:20], codeshare

def colunas_do_arquivo(arquivo_binario):
    texto = io.TextIOWrapper(arquivo_binario, encoding="utf-8-sig", newline="")
    return len(next(csv.reader(texto, delimiter=";")))

inventario = []
with zipfile.ZipFile(io.BytesIO(bytes_zip)) as z:
    membros = [m for m in z.infolist() if not m.is_dir()
               and not m.filename.startswith("__MACOSX/")
               and m.filename.lower().endswith(".csv")]
    assert len(membros) == 12, f"Esperados 12 CSVs; encontrados {len(membros)}."
    meses = set()
    for membro in sorted(membros, key=lambda m: m.filename):
        nome = PurePosixPath(membro.filename.replace("\\", "/")).name
        correspondencia = re.fullmatch(r"VRA_2021_(0[1-9]|1[0-2])\.csv", nome, flags=re.I)
        assert correspondencia, f"Nome inesperado: {nome}. Use os CSVs originais de 2021."
        mes = int(correspondencia.group(1))
        assert mes not in meses, f"Mês repetido no ZIP: {mes:02d}."
        meses.add(mes)
        assert membro.file_size > 0, f"CSV vazio: {nome}."
        with z.open(membro) as arquivo:
            quantidade = sum(1 for _ in registros_csv(arquivo, nome))
        assert quantidade > 0, f"Nenhum registro em {nome}."
        with z.open(membro) as arquivo:
            n_colunas = colunas_do_arquivo(arquivo)
        with z.open(membro) as arquivo:
            sha256 = hashlib.file_digest(arquivo, "sha256").hexdigest()
        inventario.append({"membro_zip": membro.filename, "arquivo_origem": nome,
                           "mes": mes, "colunas": n_colunas, "registros": quantidade,
                           "bytes": membro.file_size, "sha256": sha256})
assert meses == set(range(1, 13)), "Faltam meses de 2021."
inventario.sort(key=lambda item: item["mes"])
total_csv = sum(item["registros"] for item in inventario)
display(pd.DataFrame(inventario).drop(columns=["membro_zip", "sha256"]))
com_codeshare = [i["arquivo_origem"] for i in inventario if i["colunas"] == 21]
print(f"Conferência concluída: 12 meses, {total_csv:,} registros. Nenhum valor foi transformado.")
print(f"Arquivos com a coluna extra {COLUNA_EXTRA_2021!r}: {len(com_codeshare)} de 12 ->", ", ".join(com_codeshare))


     arquivo_origem  mes  colunas  registros     bytes
0   VRA_2021_01.csv    1       21      61920  18281868
1   VRA_2021_02.csv    2       21      46078  13667479
2   VRA_2021_03.csv    3       21      41701  12444242
3   VRA_2021_04.csv    4       20      30224   8455519
4   VRA_2021_05.csv    5       20      37341  10464126
5   VRA_2021_06.csv    6       20      42512  11928409
6   VRA_2021_07.csv    7       21      55555  16716773
7   VRA_2021_08.csv    8       20      56367  15837098
8   VRA_2021_09.csv    9       21      58362  17466130
9   VRA_2021_10.csv   10       20      61288  17231843
10  VRA_2021_11.csv   11       21      66649  19513413
11  VRA_2021_12.csv   12       21      74151  22010312

Conferência concluída: 12 meses, 632,148 registros. Nenhum valor foi transformado.
Arquivos com a coluna extra 'Codeshare': 7 de 12 -> VRA_2021_01.csv, VRA_2021_02.csv, VRA_2021_03.csv, VRA_2021_07.csv, VRA_2021_09.csv, VRA_2021_11.csv, VRA_2021_12.csv


## 5. Criar a tabela bruta
**Ao repetir esta célula, as tabelas deste pipeline no schema `elt_2021` serão reconstruídas.** Os schemas de 2019, 2020 e os demais schemas não são alterados.

As 20 colunas da ANAC são `TEXT`. A chave `(arquivo_origem, linha_origem)` identifica cada ocorrência, inclusive duplicatas.

Em 2021 a tabela bruta tem também a coluna `Codeshare` (`TEXT`, pode ser `NULL`). `NULL` significa que o arquivo do mês não tem a coluna; texto vazio significa que a coluna existe, mas veio em branco.


In [5]:
validacao_concluida = False
tabelas_pipeline = [
    "resumo_qualidade", "vra_conflitos_codigo", "vra_anomalias", "vra_tratada", "vra_rejeicoes",
    "vra_duplicatas_excluidas", "vra_avaliada", "vra_convertida",
    "vra_classificada", "inventario_arquivos", "vra_bruta"
]
with conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS elt_2021")
        for tabela in tabelas_pipeline:
            cur.execute(sql.SQL("DROP TABLE IF EXISTS elt_2021.{}").format(sql.Identifier(tabela)))
        campos = [sql.SQL("{} TEXT NOT NULL").format(sql.Identifier(c)) for c in colunas_esperadas]
        cur.execute(sql.SQL("""
            CREATE TABLE elt_2021.vra_bruta (
                {},
                {} TEXT NULL,
                arquivo_origem TEXT NOT NULL,
                linha_origem INTEGER NOT NULL,
                PRIMARY KEY (arquivo_origem, linha_origem)
            )
        """).format(sql.SQL(", ").join(campos), sql.Identifier(COLUNA_EXTRA_2021)))
        cur.execute("""
            CREATE TABLE elt_2021.inventario_arquivos (
                arquivo_origem TEXT PRIMARY KEY, mes INTEGER NOT NULL, colunas INTEGER NOT NULL,
                registros BIGINT NOT NULL, bytes BIGINT NOT NULL, sha256 TEXT NOT NULL
            )
        """)
print("Tabela elt_2021.vra_bruta pronta.")


Tabela elt_2021.vra_bruta pronta.


## 6. Carregar os valores originais — Load
O Python transporta os registros. Não aplica `strip`, conversão de datas, remoção de duplicatas ou filtros.

O `COPY` usa aspas em todos os campos, preservando textos vazios e valores que se pareçam com marcadores de nulo. A carga inteira é uma transação: em caso de erro, ela é desfeita.

Para os meses sem `Codeshare`, o valor é enviado como `\\N` e o `COPY` (opção `FORCE_NULL`) grava `NULL`.


In [6]:
colunas_carga = colunas_esperadas + [COLUNA_EXTRA_2021, "arquivo_origem", "linha_origem"]
comando_copy = sql.SQL(
    "COPY elt_2021.vra_bruta ({}) FROM STDIN WITH (FORMAT CSV, NULL '\\N', FORCE_NULL ({}))"
).format(sql.SQL(", ").join(sql.Identifier(c) for c in colunas_carga),
         sql.Identifier(COLUNA_EXTRA_2021)).as_string(conn)

with conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM elt_2021.vra_bruta")
        assert cur.fetchone()[0] == 0, "A tabela já contém dados. Para reconstruir a carga, execute a etapa 5."
        with zipfile.ZipFile(io.BytesIO(bytes_zip)) as z:
            for item in inventario:
                buffer = io.StringIO(newline="")
                escritor = csv.writer(buffer, quoting=csv.QUOTE_ALL, lineterminator="\n")
                with z.open(item["membro_zip"]) as arquivo:
                    for linha, campos, codeshare in registros_csv(arquivo, item["arquivo_origem"]):
                        escritor.writerow(campos + ["\\N" if codeshare is None else codeshare,
                                                    item["arquivo_origem"], linha])
                buffer.seek(0)
                cur.copy_expert(comando_copy, buffer)
                buffer.close()
                cur.execute("INSERT INTO elt_2021.inventario_arquivos VALUES (%s,%s,%s,%s,%s,%s)",
                            (item["arquivo_origem"], item["mes"], item["colunas"], item["registros"],
                             item["bytes"], item["sha256"]))
                print("Carregado:", item["arquivo_origem"])
print("Carga bruta concluída.")


Carregado: VRA_2021_01.csv
Carregado: VRA_2021_02.csv
Carregado: VRA_2021_03.csv
Carregado: VRA_2021_04.csv
Carregado: VRA_2021_05.csv
Carregado: VRA_2021_06.csv
Carregado: VRA_2021_07.csv
Carregado: VRA_2021_08.csv
Carregado: VRA_2021_09.csv
Carregado: VRA_2021_10.csv
Carregado: VRA_2021_11.csv
Carregado: VRA_2021_12.csv
Carga bruta concluída.


## 7. Conferir a carga
Comparamos a quantidade carregada por arquivo com a quantidade efetivamente lida no ZIP. Não usamos os totais de 2019 como condição de sucesso.

In [ ]:
def consultar(comando, parametros=None):
    with conn:
        with conn.cursor() as cur:
            cur.execute(comando, parametros)
            return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

carga = consultar("""
    SELECT i.arquivo_origem, i.registros AS linhas_csv, COUNT(b.arquivo_origem) AS linhas_banco
    FROM elt_2021.inventario_arquivos i
    LEFT JOIN elt_2021.vra_bruta b USING (arquivo_origem)
    GROUP BY i.arquivo_origem, i.registros
    ORDER BY i.arquivo_origem
""")
display(carga)
assert len(carga) == 12
assert (carga["linhas_csv"] == carga["linhas_banco"]).all(), "A contagem da carga diverge dos CSVs."
assert int(carga["linhas_banco"].sum()) == total_csv
print("Carga bruta conferida.")

codeshare = consultar("""
    SELECT i.colunas, COUNT(*) AS registros,
           COUNT(*) FILTER (WHERE b."Codeshare" IS NULL) AS codeshare_nulo,
           COUNT(*) FILTER (WHERE b."Codeshare" = '') AS codeshare_vazio,
           COUNT(*) FILTER (WHERE b."Codeshare" <> '') AS codeshare_preenchido
    FROM elt_2021.vra_bruta b JOIN elt_2021.inventario_arquivos i USING (arquivo_origem)
    GROUP BY i.colunas ORDER BY i.colunas
""")
display(codeshare)
assert not ((codeshare.colunas == 20) & (codeshare.codeshare_nulo != codeshare.registros)).any(), \
    "Arquivos sem a coluna Codeshare devem ter NULL."
assert not ((codeshare.colunas == 21) & (codeshare.codeshare_nulo != 0)).any(), \
    "Arquivos com a coluna Codeshare não devem ter NULL."
print("Coluna Codeshare preservada conforme o layout de cada mês.")


## 8. Transformar dentro do PostgreSQL — Transform
O SQL executa, nesta ordem:

1. Compara os **20 valores originais**, preserva a primeira ocorrência por arquivo/linha e separa as duplicatas excedentes.
2. Padroniza campos conforme o contrato: códigos em maiúsculas; nomes e descrições mantêm a grafia; vazios viram `NULL`.
3. Valida datas de verdade, inclusive dias e meses. Se um dos cinco campos obrigatórios estiver ausente ou inválido, registra **todos** os motivos de rejeição.
4. Calcula diferenças com sinal e atrasos não negativos. **Sem horário real, diferença e atraso ficam `NULL`.**
5. Marca diferenças absolutas acima de 24 horas, sem excluir ou corrigir o valor original.

Datas são `TIMESTAMP WITHOUT TIME ZONE`: não presumimos uma conversão de fuso. Anos distantes em datas reais ficam preservados quando válidos no calendário e são evidenciados pelas métricas. `ano_origem` é o ano dos arquivos, não necessariamente o ano de cada partida/chegada.

A tabela bruta permanece intacta. Campos opcionais inválidos são auditados; não tornam o voo automaticamente rejeitado.

**Ajustes de 2021 no SQL:**
- `data_segura` aceita `AAAA-MM-DD` e `DD/MM/AAAA 00:00:00` (os dois formatos de `Referência` encontrados no ano).
- A deduplicação continua usando só as 20 colunas do contrato; `Codeshare` não participa.
- Novas tabelas e contagens de auditoria pedidas pelo contrato v1.1, sem alterar a camada tratada: `vra_conflitos_codigo` (um código ICAO com mais de uma descrição), duplicatas que só aparecem depois de TRIM/UPPER e partidas previstas fora de 2021.


In [ ]:
validacao_concluida = False
SQL_TRANSFORMACAO = r"""
-- ELT 2021. Executar depois da carga bruta; nunca altera os 20 campos brutos.
CREATE OR REPLACE FUNCTION elt_2021.limpar_texto(valor TEXT)
RETURNS TEXT LANGUAGE SQL IMMUTABLE STRICT AS $$
    SELECT NULLIF(regexp_replace(valor, '^\s+|\s+$', '', 'g'), '');
$$;

CREATE OR REPLACE FUNCTION elt_2021.datahora_segura(valor TEXT)
RETURNS TIMESTAMP LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    IF valor !~ '^[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}$' THEN
        RETURN NULL;
    END IF;
    -- make_timestamp valida o calendário; não ajusta 31/02 silenciosamente.
    IF substring(valor,12,2)::INTEGER > 23 OR substring(valor,15,2)::INTEGER > 59 THEN
        RETURN NULL;
    END IF;
    RETURN make_timestamp(substring(valor,7,4)::INTEGER, substring(valor,4,2)::INTEGER,
                          substring(valor,1,2)::INTEGER, substring(valor,12,2)::INTEGER,
                          substring(valor,15,2)::INTEGER, 0);
EXCEPTION WHEN datetime_field_overflow OR invalid_datetime_format OR numeric_value_out_of_range THEN
    RETURN NULL;
END;
$$;

CREATE OR REPLACE FUNCTION elt_2021.data_segura(valor TEXT)
RETURNS DATE LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    -- 2021 traz dois formatos: AAAA-MM-DD e DD/MM/AAAA 00:00:00.
    IF valor ~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN
        RETURN make_date(substring(valor,1,4)::INTEGER, substring(valor,6,2)::INTEGER,
                         substring(valor,9,2)::INTEGER);
    END IF;
    IF valor ~ '^[0-9]{2}/[0-9]{2}/[0-9]{4} 00:00:00$' THEN
        RETURN make_date(substring(valor,7,4)::INTEGER, substring(valor,4,2)::INTEGER,
                         substring(valor,1,2)::INTEGER);
    END IF;
    RETURN NULL;
EXCEPTION WHEN datetime_field_overflow OR invalid_datetime_format OR numeric_value_out_of_range THEN
    RETURN NULL;
END;
$$;

CREATE OR REPLACE FUNCTION elt_2021.inteiro_seguro(valor TEXT)
RETURNS INTEGER LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    IF valor !~ '^[0-9]+$' THEN RETURN NULL; END IF;
    RETURN valor::INTEGER;
EXCEPTION WHEN numeric_value_out_of_range THEN RETURN NULL;
END;
$$;

DROP TABLE IF EXISTS elt_2021.resumo_qualidade;
DROP TABLE IF EXISTS elt_2021.vra_conflitos_codigo;
DROP TABLE IF EXISTS elt_2021.vra_anomalias;
DROP TABLE IF EXISTS elt_2021.vra_tratada;
DROP TABLE IF EXISTS elt_2021.vra_rejeicoes;
DROP TABLE IF EXISTS elt_2021.vra_duplicatas_excluidas;
DROP TABLE IF EXISTS elt_2021.vra_avaliada;
DROP TABLE IF EXISTS elt_2021.vra_convertida;
DROP TABLE IF EXISTS elt_2021.vra_classificada;

CREATE TABLE elt_2021.vra_classificada AS
SELECT b.*,
       ROW_NUMBER() OVER w AS ocorrencia_num,
       FIRST_VALUE(arquivo_origem) OVER w AS arquivo_preservado,
       FIRST_VALUE(linha_origem) OVER w AS linha_preservada
FROM elt_2021.vra_bruta b
WINDOW w AS (
    PARTITION BY "Sigla ICAO Empresa Aérea" COLLATE "C", "Empresa Aérea" COLLATE "C", "Número Voo" COLLATE "C", "Código DI" COLLATE "C", "Código Tipo Linha" COLLATE "C", "Modelo Equipamento" COLLATE "C", "Número de Assentos" COLLATE "C", "Sigla ICAO Aeroporto Origem" COLLATE "C", "Descrição Aeroporto Origem" COLLATE "C", "Partida Prevista" COLLATE "C", "Partida Real" COLLATE "C", "Sigla ICAO Aeroporto Destino" COLLATE "C", "Descrição Aeroporto Destino" COLLATE "C", "Chegada Prevista" COLLATE "C", "Chegada Real" COLLATE "C", "Situação Voo" COLLATE "C", "Justificativa" COLLATE "C", "Referência" COLLATE "C", "Situação Partida" COLLATE "C", "Situação Chegada" COLLATE "C"
    ORDER BY arquivo_origem COLLATE "C", linha_origem
);

CREATE TABLE elt_2021.vra_duplicatas_excluidas AS
SELECT * FROM elt_2021.vra_classificada WHERE ocorrencia_num > 1;

-- Só após identificar duplicatas EXATAS é feita a padronização.
CREATE TABLE elt_2021.vra_convertida AS
SELECT UPPER(elt_2021.limpar_texto("Sigla ICAO Empresa Aérea")) AS companhia_icao,
       elt_2021.limpar_texto("Empresa Aérea") AS companhia_nome,
       elt_2021.limpar_texto("Número Voo") AS numero_voo,
       UPPER(elt_2021.limpar_texto("Código DI")) AS codigo_di,
       UPPER(elt_2021.limpar_texto("Código Tipo Linha")) AS tipo_linha,
       UPPER(elt_2021.limpar_texto("Modelo Equipamento")) AS modelo_equipamento,
       elt_2021.inteiro_seguro(elt_2021.limpar_texto("Número de Assentos")) AS numero_assentos,
       UPPER(elt_2021.limpar_texto("Sigla ICAO Aeroporto Origem")) AS aeroporto_origem,
       elt_2021.limpar_texto("Descrição Aeroporto Origem") AS descricao_origem,
       elt_2021.datahora_segura(elt_2021.limpar_texto("Partida Prevista")) AS partida_prevista,
       elt_2021.datahora_segura(elt_2021.limpar_texto("Partida Real")) AS partida_real,
       UPPER(elt_2021.limpar_texto("Sigla ICAO Aeroporto Destino")) AS aeroporto_destino,
       elt_2021.limpar_texto("Descrição Aeroporto Destino") AS descricao_destino,
       elt_2021.datahora_segura(elt_2021.limpar_texto("Chegada Prevista")) AS chegada_prevista,
       elt_2021.datahora_segura(elt_2021.limpar_texto("Chegada Real")) AS chegada_real,
       UPPER(elt_2021.limpar_texto("Situação Voo")) AS situacao_voo,
       elt_2021.limpar_texto("Justificativa") AS justificativa,
       elt_2021.data_segura(elt_2021.limpar_texto("Referência")) AS referencia,
       UPPER(elt_2021.limpar_texto("Situação Partida")) AS situacao_partida,
       UPPER(elt_2021.limpar_texto("Situação Chegada")) AS situacao_chegada,
       arquivo_origem, linha_origem, 2021::INTEGER AS ano_origem
FROM elt_2021.vra_classificada
WHERE ocorrencia_num = 1;

CREATE TABLE elt_2021.vra_avaliada AS
SELECT c.*, ARRAY_REMOVE(ARRAY[
    CASE WHEN companhia_icao IS NULL THEN 'COMPANHIA_AUSENTE' END,
    CASE WHEN aeroporto_origem IS NULL THEN 'AEROPORTO_ORIGEM_AUSENTE' END,
    CASE WHEN aeroporto_destino IS NULL THEN 'AEROPORTO_DESTINO_AUSENTE' END,
    CASE WHEN partida_prevista IS NULL THEN
        CASE WHEN elt_2021.limpar_texto(b."Partida Prevista") IS NULL
             THEN 'PARTIDA_PREVISTA_AUSENTE' ELSE 'PARTIDA_PREVISTA_INVALIDA' END END,
    CASE WHEN chegada_prevista IS NULL THEN
        CASE WHEN elt_2021.limpar_texto(b."Chegada Prevista") IS NULL
             THEN 'CHEGADA_PREVISTA_AUSENTE' ELSE 'CHEGADA_PREVISTA_INVALIDA' END END
], NULL) AS motivos_rejeicao
FROM elt_2021.vra_convertida c
JOIN elt_2021.vra_bruta b USING (arquivo_origem, linha_origem);

-- Preserva todos os campos originais e todos os motivos (uma linha por registro).
CREATE TABLE elt_2021.vra_rejeicoes AS
SELECT b.*, a.motivos_rejeicao
FROM elt_2021.vra_bruta b
JOIN elt_2021.vra_avaliada a USING (arquivo_origem, linha_origem)
WHERE cardinality(a.motivos_rejeicao) > 0;

CREATE TABLE elt_2021.vra_tratada AS
WITH medidas AS (
    SELECT a.*,
           EXTRACT(EPOCH FROM (partida_real - partida_prevista))/60 AS diferenca_partida_min,
           EXTRACT(EPOCH FROM (chegada_real - chegada_prevista))/60 AS diferenca_chegada_min
    FROM elt_2021.vra_avaliada a
    WHERE cardinality(motivos_rejeicao) = 0
)
SELECT *,
       CASE WHEN diferenca_partida_min IS NOT NULL THEN GREATEST(diferenca_partida_min,0) END AS atraso_partida_min,
       CASE WHEN diferenca_chegada_min IS NOT NULL THEN GREATEST(diferenca_chegada_min,0) END AS atraso_chegada_min,
       COALESCE(ABS(diferenca_partida_min)>1440,FALSE) AS partida_atipica_flag,
       COALESCE(ABS(diferenca_chegada_min)>1440,FALSE) AS chegada_atipica_flag
FROM medidas;

ALTER TABLE elt_2021.vra_tratada DROP COLUMN motivos_rejeicao;
ALTER TABLE elt_2021.vra_tratada ADD PRIMARY KEY (arquivo_origem, linha_origem);
ALTER TABLE elt_2021.vra_rejeicoes ADD PRIMARY KEY (arquivo_origem, linha_origem);

-- Campos opcionais inválidos não excluem o voo; ficam registrados para auditoria.
CREATE TABLE elt_2021.vra_anomalias AS
SELECT c.arquivo_origem, c.linha_origem, v.campo, v.valor_original,
       'VALOR_OPCIONAL_INVALIDO'::TEXT AS motivo
FROM elt_2021.vra_convertida c
JOIN elt_2021.vra_bruta b USING (arquivo_origem, linha_origem)
CROSS JOIN LATERAL (VALUES
    ('Partida Real', b."Partida Real", c.partida_real IS NULL),
    ('Chegada Real', b."Chegada Real", c.chegada_real IS NULL),
    ('Número de Assentos', b."Número de Assentos", c.numero_assentos IS NULL),
    ('Referência', b."Referência", c.referencia IS NULL)
) AS v(campo, valor_original, falhou)
WHERE elt_2021.limpar_texto(v.valor_original) IS NOT NULL AND v.falhou;

INSERT INTO elt_2021.vra_anomalias
SELECT arquivo_origem, linha_origem, 'Referência', referencia::TEXT, 'REFERENCIA_FORA_DE_2021'
FROM elt_2021.vra_convertida
WHERE referencia IS NOT NULL AND EXTRACT(YEAR FROM referencia) <> 2021;

-- Contrato v1.1, seção 4: um código ICAO não pode ter descrições diferentes
-- numa dimensão compartilhada. Aqui só registramos; a decisão é do grupo.
CREATE TABLE elt_2021.vra_conflitos_codigo AS
WITH pares AS (
    SELECT 'companhia' AS tipo, companhia_icao AS codigo, companhia_nome AS descricao FROM elt_2021.vra_tratada
    UNION ALL
    SELECT 'aeroporto', aeroporto_origem, descricao_origem FROM elt_2021.vra_tratada
    UNION ALL
    SELECT 'aeroporto', aeroporto_destino, descricao_destino FROM elt_2021.vra_tratada
)
SELECT tipo, codigo, descricao, COUNT(*) AS ocorrencias
FROM pares
WHERE (tipo, codigo) IN (
    SELECT tipo, codigo FROM pares GROUP BY tipo, codigo
    HAVING COUNT(DISTINCT COALESCE(descricao, '<NULL>')) > 1
)
GROUP BY tipo, codigo, descricao
ORDER BY tipo, codigo, descricao;

CREATE TABLE elt_2021.resumo_qualidade AS
SELECT
    (SELECT COUNT(*) FROM elt_2021.vra_bruta) AS bruta,
    (SELECT COUNT(*) FROM elt_2021.vra_duplicatas_excluidas) AS duplicatas_excedentes,
    (SELECT COUNT(*) FROM elt_2021.vra_rejeicoes) AS rejeicoes,
    (SELECT COUNT(*) FROM elt_2021.vra_tratada) AS tratada,
    (SELECT COUNT(*) FROM elt_2021.vra_tratada WHERE partida_atipica_flag) AS partidas_atipicas,
    (SELECT COUNT(*) FROM elt_2021.vra_tratada WHERE chegada_atipica_flag) AS chegadas_atipicas,
    (SELECT COUNT(*) FROM elt_2021.vra_tratada WHERE partida_real IS NULL) AS partidas_reais_nulas,
    (SELECT COUNT(*) FROM elt_2021.vra_tratada WHERE chegada_real IS NULL) AS chegadas_reais_nulas,
    (SELECT COUNT(*) FROM elt_2021.vra_anomalias) AS ocorrencias_anomalias,
    -- Contrato v1.1, seção 5: duplicatas só após TRIM/UPPER são informadas, não excluídas.
    (SELECT COALESCE(SUM(n - 1), 0) FROM (
        SELECT COUNT(*) AS n FROM elt_2021.vra_convertida
        GROUP BY companhia_icao, companhia_nome, numero_voo, codigo_di, tipo_linha,
                 modelo_equipamento, numero_assentos, aeroporto_origem, descricao_origem,
                 partida_prevista, partida_real, aeroporto_destino, descricao_destino,
                 chegada_prevista, chegada_real, situacao_voo, justificativa, referencia,
                 situacao_partida, situacao_chegada
        HAVING COUNT(*) > 1) g) AS duplicatas_apos_padronizacao,
    -- Contrato v1.1, seção 4: registrar diferença entre ano do arquivo e ano da partida prevista.
    (SELECT COUNT(*) FROM elt_2021.vra_tratada
      WHERE EXTRACT(YEAR FROM partida_prevista) <> ano_origem) AS partidas_previstas_fora_do_ano,
    (SELECT COUNT(DISTINCT (tipo, codigo)) FROM elt_2021.vra_conflitos_codigo) AS codigos_com_conflito,
    CURRENT_TIMESTAMP AS executado_em;
"""

with conn:
    with conn.cursor() as cur:
        cur.execute(SQL_TRANSFORMACAO)
print("Camada tratada e tabelas de auditoria criadas por SQL.")

## 9. Validar o resultado
A primeira igualdade é a conferência principal:

**bruta = duplicatas excedentes + rejeições após deduplicação + tratada**

Também verificamos campos obrigatórios, unicidade da origem, preservação da carga e cálculo dos atrasos. Os testes de datas incluem ano bissexto, dia impossível e horário inválido.

As anomalias opcionais e os motivos de rejeição não são novas linhas de voos: um registro pode ter mais de um motivo.

In [ ]:
resumo = consultar("SELECT * FROM elt_2021.resumo_qualidade")
display(resumo.T.rename(columns={0: "Resultado"}))
r = resumo.iloc[0]
assert int(r["bruta"]) == total_csv, "A camada bruta diverge do ZIP."
assert r["bruta"] == r["duplicatas_excedentes"] + r["rejeicoes"] + r["tratada"], "Falha na reconciliação."

checagens = consultar("""
    SELECT
      NOT EXISTS (
        SELECT 1 FROM elt_2021.vra_tratada
        WHERE companhia_icao IS NULL OR aeroporto_origem IS NULL OR aeroporto_destino IS NULL
           OR partida_prevista IS NULL OR chegada_prevista IS NULL
      ) AS obrigatorios_ok,
      NOT EXISTS (
        SELECT 1 FROM elt_2021.vra_tratada t JOIN elt_2021.vra_rejeicoes r
        USING (arquivo_origem, linha_origem)
      ) AS conjuntos_separados,
      NOT EXISTS (
        SELECT 1 FROM elt_2021.vra_tratada t JOIN elt_2021.vra_duplicatas_excluidas d
        USING (arquivo_origem, linha_origem)
      ) AS sem_excedentes_na_tratada,
      NOT EXISTS (
        SELECT 1 FROM elt_2021.vra_tratada
        WHERE atraso_partida_min IS DISTINCT FROM
          CASE WHEN partida_real IS NOT NULL THEN GREATEST(EXTRACT(EPOCH FROM (partida_real-partida_prevista))/60,0) END
        OR atraso_chegada_min IS DISTINCT FROM
          CASE WHEN chegada_real IS NOT NULL THEN GREATEST(EXTRACT(EPOCH FROM (chegada_real-chegada_prevista))/60,0) END
      ) AS atrasos_ok,
      elt_2021.datahora_segura('29/02/2020 23:59') = TIMESTAMP '2020-02-29 23:59' AS bissexto_ok,
      elt_2021.datahora_segura('29/02/2019 23:59') IS NULL AS dia_invalido_rejeitado,
      elt_2021.datahora_segura('30/04/2020 24:01') IS NULL AS hora_invalida_rejeitada,
      elt_2021.datahora_segura('31/04/2020 12:00') IS NULL AS abril_invalido_rejeitado,
      elt_2021.data_segura('2021-04-30') = DATE '2021-04-30' AS referencia_iso_ok,
      elt_2021.data_segura('07/01/2021 00:00:00') = DATE '2021-01-07' AS referencia_br_ok,
      elt_2021.data_segura('31/04/2021 00:00:00') IS NULL AS referencia_invalida_rejeitada,
      NOT EXISTS (SELECT 1 FROM elt_2021.vra_anomalias WHERE campo = 'Referência')
        AS referencias_convertidas,
      elt_2021.inteiro_seguro('-1') IS NULL AS assento_negativo_rejeitado,
      elt_2021.inteiro_seguro('99999999999999999999') IS NULL AS inteiro_grande_tratado
""")
display(checagens.T.rename(columns={0: "Passou?"}))
assert checagens.iloc[0].all(), "Uma validação falhou; consulte a tabela acima."

# As contagens individuais também precisam continuar iguais após transformar.
carga_final = consultar("""
    SELECT arquivo_origem, COUNT(*) AS total FROM elt_2021.vra_bruta
    GROUP BY arquivo_origem ORDER BY arquivo_origem
""")
assert dict(zip(carga_final.arquivo_origem, carga_final.total)) == {
    item["arquivo_origem"]: item["registros"] for item in inventario
}
validacao_concluida = True
print("ELT 2021: carga, reconciliação e verificações de qualidade passaram nesta execução.")

## 10. Evidências para explicar no relatório
Leia os resultados: quantos registros foram descartados como repetição exata, quantos ficaram em rejeições e quantos foram tratados? Uma rejeição com dois campos faltantes aparece em dois motivos, mas conta como **um registro rejeitado**.

Estes resultados ainda precisam ser comparados com o ETL de 2021 do colega, usando os mesmos CSVs e as mesmas regras. Igualdade dos totais é necessária, mas a comparação ideal inclui os valores por `arquivo_origem + linha_origem`.

In [10]:
CONSULTAS_EVIDENCIAS = {
    "mensal": """
      SELECT i.arquivo_origem, i.registros AS bruta,
        (SELECT COUNT(*) FROM elt_2021.vra_duplicatas_excluidas d WHERE d.arquivo_origem=i.arquivo_origem) AS duplicatas,
        (SELECT COUNT(*) FROM elt_2021.vra_rejeicoes r WHERE r.arquivo_origem=i.arquivo_origem) AS rejeicoes,
        (SELECT COUNT(*) FROM elt_2021.vra_tratada t WHERE t.arquivo_origem=i.arquivo_origem) AS tratada
      FROM elt_2021.inventario_arquivos i ORDER BY i.arquivo_origem
    """,
    "motivos_rejeicao": """
      SELECT motivo, COUNT(*) AS quantidade
      FROM elt_2021.vra_rejeicoes CROSS JOIN LATERAL unnest(motivos_rejeicao) AS m(motivo)
      GROUP BY motivo ORDER BY quantidade DESC, motivo
    """,
    "situacao_voo": """
      SELECT situacao_voo, COUNT(*) AS quantidade FROM elt_2021.vra_tratada
      GROUP BY situacao_voo ORDER BY quantidade DESC
    """,
    "atipicos_amostra": """
      SELECT arquivo_origem, linha_origem, numero_voo, partida_prevista, partida_real,
             chegada_prevista, chegada_real, diferenca_partida_min, diferenca_chegada_min
      FROM elt_2021.vra_tratada WHERE partida_atipica_flag OR chegada_atipica_flag
      ORDER BY GREATEST(ABS(diferenca_partida_min),ABS(diferenca_chegada_min)) DESC,
               arquivo_origem, linha_origem LIMIT 10
    """,
    "layout_por_arquivo": """
      SELECT arquivo_origem, colunas, registros FROM elt_2021.inventario_arquivos ORDER BY mes
    """,
    "conflitos_codigo": """
      SELECT * FROM elt_2021.vra_conflitos_codigo ORDER BY tipo, codigo, descricao
    """,
    "duplicatas_amostra": """
      SELECT arquivo_origem, linha_origem, arquivo_preservado, linha_preservada
      FROM elt_2021.vra_duplicatas_excluidas ORDER BY arquivo_origem, linha_origem LIMIT 10
    """
}
evidencias = {}
for titulo, consulta in CONSULTAS_EVIDENCIAS.items():
    print("\n" + titulo.replace("_", " ").upper())
    evidencias[titulo] = consultar(consulta)
    display(evidencias[titulo])
mensal = evidencias["mensal"]
assert (mensal.bruta == mensal.duplicatas + mensal.rejeicoes + mensal.tratada).all()


MENSAL


     arquivo_origem  bruta  duplicatas  rejeicoes  tratada
0   VRA_2021_01.csv  61920           0       3505    58415
1   VRA_2021_02.csv  46078           0       2486    43592
2   VRA_2021_03.csv  41701           0       2326    39375
3   VRA_2021_04.csv  30224           0       1250    28974
4   VRA_2021_05.csv  37341           0       1739    35602
5   VRA_2021_06.csv  42512           0       1680    40832
6   VRA_2021_07.csv  55555           0       2335    53220
7   VRA_2021_08.csv  56367           0       1517    54850
8   VRA_2021_09.csv  58362           0       1881    56481
9   VRA_2021_10.csv  61288           0       1860    59428
10  VRA_2021_11.csv  66649           0       2585    64064
11  VRA_2021_12.csv  74151           0       2347    71804


MOTIVOS REJEICAO


                     motivo  quantidade
0  CHEGADA_PREVISTA_AUSENTE       25511
1  PARTIDA_PREVISTA_AUSENTE       25511


SITUACAO VOO


    situacao_voo  quantidade
0      REALIZADO      580498
1      CANCELADO       25962
2  NÃO INFORMADO         177

... (4 outputs left out)


CONFLITOS CODIGO


         tipo codigo                                          descricao  \
0   aeroporto   KAEX  ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...   
1   aeroporto   KAEX  ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...   
2   aeroporto   SWHP              OLHOS D´ÁGUA - ÁGUA BOA - MG - BRASIL   
3   aeroporto   SWHP              OLHOS D´ÁGUA - ÁGUA BOA - MT - BRASIL   
4   aeroporto   SYEC  EUGENE F. CORREIRA INTERNATIONAL - OGLE -  - G...   
5   aeroporto   SYEC   EUGENE F. CORREIRA INTERNATIONAL - OGLE - GUIANA   
6   companhia    CQB                              APUÍ TÁXI AÉREO LTDA.   
7   companhia    CQB                                APUÍ TÁXI AÉREO S/A   
8   companhia    DTA     TAAG LINHAS AÉREAS DA ANGOLA (ANGOLA AIRLINES)   
9   companhia    DTA     TAAG LINHAS AÉREAS DE ANGOLA (ANGOLA AIRLINES)   
10  companhia    ETH                      ETHIOPIAN AIRLINES ENTERPRISE   
11  companhia    ETH                           ETHIOPIAN AIRLINES GROUP   
12  companhia    ETR     


DUPLICATAS AMOSTRA


Empty DataFrame
Columns: [arquivo_origem, linha_origem, arquivo_preservado, linha_preservada]
Index: []

## 11. Gerar e baixar as evidências
Esta etapa cria `evidencias_elt_2021.zip` com resumo, inventário e hashes dos arquivos, contagens por mês, rejeições, duplicatas, anomalias e o SQL executado.

A tabela tratada completa fica no PostgreSQL. Para incluí-la no ZIP, altere `EXPORTAR_TRATADA_COMPLETA` para `True`. O arquivo será maior; não é necessário colocar a base inteira no GitHub.

Só gere a entrega final após a etapa 9 passar.

In [11]:
from pathlib import Path
import json
import platform

assert globals().get("validacao_concluida", False), "Execute a validação da etapa 9 primeiro."
EXPORTAR_TRATADA_COMPLETA = False
pasta_saida = Path("/content/resultados_elt_2021")
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivos_saida = []

def guardar_texto(nome, conteudo):
    destino = pasta_saida / nome
    destino.write_text(conteudo, encoding="utf-8")
    arquivos_saida.append(destino)

def exportar_consulta(nome, consulta):
    destino = pasta_saida / nome
    with conn:
        with conn.cursor() as cur:
            with destino.open("w", encoding="utf-8-sig", newline="") as arquivo:
                cur.copy_expert("COPY (" + consulta.strip().rstrip(";") +
                                ") TO STDOUT WITH (FORMAT CSV, HEADER TRUE)", arquivo)
    arquivos_saida.append(destino)

guardar_texto("transformacoes_elt_2021.sql", SQL_TRANSFORMACAO)
exportar_consulta("resumo_qualidade_2021.csv", "SELECT * FROM elt_2021.resumo_qualidade")
exportar_consulta("inventario_arquivos_2021.csv", "SELECT * FROM elt_2021.inventario_arquivos ORDER BY mes")
for nome, consulta in CONSULTAS_EVIDENCIAS.items():
    exportar_consulta(nome + "_2021.csv", consulta)
for tabela in ["vra_rejeicoes", "vra_duplicatas_excluidas", "vra_anomalias", "vra_conflitos_codigo"]:
    ordem = "tipo, codigo, descricao" if tabela == "vra_conflitos_codigo" else "arquivo_origem, linha_origem"
    exportar_consulta(tabela + "_2021.csv",
                     "SELECT * FROM elt_2021." + tabela + " ORDER BY " + ordem)
if EXPORTAR_TRATADA_COMPLETA:
    exportar_consulta("vra_tratada_2021.csv",
                     "SELECT * FROM elt_2021.vra_tratada ORDER BY arquivo_origem, linha_origem")

versao_postgres = consultar("SELECT version() AS versao").iloc[0]["versao"]
metadados = {
    "autor_adaptacao": "Murilo Acioli",
    "base": "notebooks/elt/elt_2020.ipynb (Gustavo José Arraes de Queiroz)",
    "fonte": "https://siros.anac.gov.br/siros/registros/diversos/vra/2021/",
    "pipeline": "ELT", "ano_origem": ANO, "versao_pipeline": VERSAO_PIPELINE,
    "inicio_utc": inicio_execucao, "fim_utc": datetime.now(timezone.utc).isoformat(),
    "zip_origem": nome_zip, "sha256_zip": sha256_zip, "arquivos": inventario,
    "python": platform.python_version(), "postgresql": versao_postgres,
    "validacao": "Verificações da etapa 9 passaram; reconciliação mensal da etapa 10 passou.",
    "comparacao_etl": "Pendente de comparação com o resultado do colega."
}
guardar_texto("metadados_execucao_2021.json", json.dumps(metadados, ensure_ascii=False, indent=2))
guardar_texto("LEIA_ME.md", f"""# Evidências ELT 2021
Responsável: Murilo Acioli. Base: notebook ELT 2020 do grupo (Gustavo José Arraes de Queiroz).

Execução iniciada em {inicio_execucao} (UTC).
ZIP original: {nome_zip}
SHA-256: {sha256_zip}

Reconciliação: {int(r['bruta'])} brutos = {int(r['duplicatas_excedentes'])} duplicatas excedentes
+ {int(r['rejeicoes'])} rejeições + {int(r['tratada'])} tratados.

O SQL transforma somente após a carga dos valores originais no PostgreSQL.
Duplicatas são comparadas nas 20 colunas do contrato antes da padronização (Codeshare não entra).
7 dos 12 arquivos de 2021 têm a coluna extra Codeshare; ela foi preservada na camada bruta.
Sem horário real, o atraso permanece nulo.
Diferenças acima de 24 horas são sinalizadas e preservadas.
Rejeições conservam todos os valores originais e todos os motivos.
O inventário contém quantidade e SHA-256 de cada CSV, para comparar com o ETL.

Escopo: camada tratada e auditoria. Dimensões e fato compartilhadas não foram criadas.
A comparação completa entre ETL e ELT permanece pendente.
transformacoes_elt_2021.sql pressupõe a carga bruta criada pelo notebook.

Para o GitHub, selecione notebook, SQL, resumo e explicação dos resultados.
Não é necessário enviar os CSVs brutos ou a tabela tratada completa.
""")

caminho_zip = Path("/content/evidencias_elt_2021.zip")
with zipfile.ZipFile(caminho_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for arquivo in arquivos_saida:
        z.write(arquivo, arcname=arquivo.name)
print("Evidências geradas:", caminho_zip.name)
files.download(str(caminho_zip))

Evidências geradas: evidencias_elt_2021.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Salvar a contribuição no GitHub
Depois de executar e conferir os resultados:

1. Salve este notebook executado em **`notebooks/elt/elt_2021.ipynb`**, na branch **`feat-elt`**.
2. Commit sugerido: **`feat: implementa pipeline ELT de 2021 com auditoria em SQL`**.
3. Do ZIP de evidências, acrescente o SQL em `sql/transformacoes/elt_2021.sql`, os CSVs em `resultados/` (sufixo `_elt_2021`) e a explicação em `relatorio/relatorio_elt_2021.md`.
4. Commit sugerido: **`docs: registra execução e resultados do ELT de 2021`**.
5. Compare com o ETL de 2021 (Arthur): mesmos hashes dos CSVs, critérios de deduplicação, rejeições, totais e valores por registro. Depois abram o pull request para revisão do grupo.

**Para explicar:** o Python só leu e carregou os dados; o PostgreSQL fez a transformação. A tabela bruta permite auditar cada resultado. Duplicatas excedentes e rejeições têm destinos separados, e a reconciliação impede que uma linha desapareça sem explicação.

**Ainda depende da integração do grupo:** dimensões e `fato_voo` conjuntas, análises finais, comparação ETL × ELT e relatório/apresentação completos.

Não suba senhas, arquivos `.env` ou os CSVs brutos. Preserve o crédito aos notebooks-base do grupo.
